# Faz 03 — Sentence-BERT + CLIP Embedding

Bu notebook Colab'de **GPU runtime** ile çalışır.  `Runtime → Change runtime type → T4 GPU`.

İki şey üretiyoruz:
1. **Sentence-BERT** text embedding'leri (13,575 ürün × 384-d, ~3 dk)
2. **CLIP** image embedding'leri (13,575 ürün × 512-d, ~30-60 dk — image download dahil)

Çıktılar Drive'a kaydediliyor, indirip `results/embeddings/` klasörüne koyulacak.

transformers>=4.30,<5.0

## 1. Kurulum

In [1]:
!pip install -q sentence-transformers transformers torch pillow requests
print("✓ Paketler kuruldu")

✓ Paketler kuruldu


In [2]:
# GPU kontrolü
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
else:
    print('GPU yok! Runtime → Change runtime type → T4 GPU seç')

CUDA available: True
Device: Tesla T4


In [3]:
# Drive bağla
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/data-mining-hw'
EMBED_DIR = f'{PROJECT_DIR}/embeddings'
os.makedirs(EMBED_DIR, exist_ok=True)
print(f'Embeddings klasörü: {EMBED_DIR}')

Mounted at /content/drive
Embeddings klasörü: /content/drive/MyDrive/data-mining-hw/embeddings


In [4]:
# HF login (dataset okumak için lazım değil, public)
try:
    from google.colab import userdata
    from huggingface_hub import login
    token = userdata.get('HF_TOKEN')
    login(token=token)
    print('✓ HF login')
except Exception as e:
    print(f'HF login skip: {e} (public dataset için sorun değil)')

✓ HF login


## 2. Veriyi HF'ten yükle

In [5]:
import pandas as pd
from huggingface_hub import hf_hub_download

DATASET_REPO = 'oyku-tugana/amazon-musical-instruments-2018-2023-5core'

print('items.parquet indiriliyor...')
items_path = hf_hub_download(
    repo_id=DATASET_REPO,
    filename='items/train-00000-of-00001.parquet',
    repo_type='dataset',
)
items = pd.read_parquet(items_path)
print(f'✓ items: {items.shape}')
print(f'Kolonlar: {list(items.columns)}')

items.parquet indiriliyor...


items/train-00000-of-00001.parquet:   0%|          | 0.00/10.4M [00:00<?, ?B/s]

✓ items: (13575, 14)
Kolonlar: ['parent_asin', 'title', 'description', 'features', 'main_category', 'categories', 'store', 'price', 'average_rating', 'rating_number', 'image_url', 'has_image', 'has_description', 'has_price']


## 3. Item text representasyonu

In [6]:
def build_item_text(row):
    parts = []
    for col in ['title', 'description', 'features', 'categories', 'store']:
        v = row[col]
        if isinstance(v, str) and v.strip():
            parts.append(v)
    return ' '.join(parts)

items['text'] = items.apply(build_item_text, axis=1)
print(f'Text hazır. Avg uzunluk: {items["text"].str.len().mean():.0f}')

Text hazır. Avg uzunluk: 1252


## 4. Sentence-BERT Embedding

In [7]:
from sentence_transformers import SentenceTransformer
import time, numpy as np

print('Model yükleniyor (all-MiniLM-L6-v2)...')
sbert_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
print(f'Model: dim={sbert_model.get_sentence_embedding_dimension()}')

Model yükleniyor (all-MiniLM-L6-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model: dim=384


/tmp/ipykernel_1805/1079171614.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'Model: dim={sbert_model.get_sentence_embedding_dimension()}')


In [8]:
# Tüm ürün text'lerini embed et
print(f'{len(items):,} ürün için S-BERT embedding üretiliyor...')
t0 = time.time()
sbert_embeddings = sbert_model.encode(
    items['text'].fillna('').tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # L2 normalize, cosine için
)
print(f'\n✓ Tamam: {sbert_embeddings.shape}, süre: {time.time()-t0:.1f}s')

13,575 ürün için S-BERT embedding üretiliyor...


Batches:   0%|          | 0/107 [00:00<?, ?it/s]


✓ Tamam: (13575, 384), süre: 42.1s


In [9]:
# Drive'a kaydet
import pickle
np.save(f'{EMBED_DIR}/sbert_embeddings.npy', sbert_embeddings)
with open(f'{EMBED_DIR}/sbert_item_ids.pkl', 'wb') as f:
    pickle.dump(items['parent_asin'].tolist(), f)

print(f'✓ Kaydedildi:')
print(f'  {EMBED_DIR}/sbert_embeddings.npy ({os.path.getsize(EMBED_DIR + "/sbert_embeddings.npy")/1e6:.1f} MB)')
print(f'  {EMBED_DIR}/sbert_item_ids.pkl')

✓ Kaydedildi:
  /content/drive/MyDrive/data-mining-hw/embeddings/sbert_embeddings.npy (20.9 MB)
  /content/drive/MyDrive/data-mining-hw/embeddings/sbert_item_ids.pkl


## 5. CLIP Image Embedding

Bu kısım yavaş — her ürünün görselini indirip işliyor. 13K+ görsel için 30-60 dakika sürebilir.

Tasarım:
- Parallel download (20 thread)
- Batch CLIP encoding (her 256 görsel)
- Her 1000 ürünün sonunda checkpoint kaydet (Colab session düşerse kaldığımız yerden başlayalım)

In [10]:
from transformers import CLIPProcessor, CLIPModel

print('CLIP model yükleniyor...')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to('cuda')
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()
print(f'CLIP image dim: {clip_model.config.projection_dim}')

CLIP model yükleniyor...


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP image dim: 512


In [11]:
import requests
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed

# Sadece image_url'i olan ürünler
items_with_url = items[items['image_url'].notna() & (items['image_url'] != '')].copy().reset_index(drop=True)
print(f'Image URL\'si olan ürün: {len(items_with_url):,} / {len(items):,}')

def download_image(idx_url):
    idx, url = idx_url
    try:
        r = requests.get(url, timeout=10, headers={'User-Agent': 'Mozilla/5.0'})
        r.raise_for_status()
        img = Image.open(BytesIO(r.content)).convert('RGB')
        return idx, img, None
    except Exception as e:
        return idx, None, str(e)[:50]

print('İlk görseli test ediyoruz...')
test_idx, test_img, test_err = download_image((0, items_with_url.iloc[0]['image_url']))
if test_img:
    print(f'✓ İlk görsel başarıyla indirildi: {test_img.size}')
else:
    print(f'✗ İlk görsel hatası: {test_err}')

Image URL'si olan ürün: 13,572 / 13,575
İlk görseli test ediyoruz...
✓ İlk görsel başarıyla indirildi: (473, 500)


In [12]:
# Resume kontrolü: zaten yapılmış embedding var mı?
checkpoint_path = f'{EMBED_DIR}/clip_checkpoint.npz'

if os.path.exists(checkpoint_path):
    cp = np.load(checkpoint_path, allow_pickle=True)
    clip_embeddings_partial = cp['embeddings']
    successful_ids = list(cp['item_ids'])
    failed_ids = list(cp['failed_ids'])
    print(f'Checkpoint bulundu: {len(successful_ids):,} başarılı, {len(failed_ids):,} başarısız')
    print(f'Kaldığımız yerden devam ediyoruz...')
    done_ids = set(successful_ids) | set(failed_ids)
else:
    clip_embeddings_partial = np.zeros((0, clip_model.config.projection_dim), dtype=np.float32)
    successful_ids, failed_ids = [], []
    done_ids = set()
    print('Sıfırdan başlıyoruz')

# Yapılacak ürünler
to_process = items_with_url[~items_with_url['parent_asin'].isin(done_ids)].reset_index(drop=True)
print(f'İşlenecek: {len(to_process):,}')

Sıfırdan başlıyoruz
İşlenecek: 13,572


In [15]:
# Ana loop — parallel download + batch CLIP encoding
BATCH_SIZE = 64  # CLIP batch
DL_WORKERS = 20  # Parallel download thread sayısı
SAVE_EVERY = 1000  # Her N ürünün sonunda checkpoint

new_embeddings = []
new_success_ids = []
new_failed_ids = []

t0 = time.time()
processed_count = 0

# Chunklara böl (her chunk içinde parallel download + batch CLIP)
CHUNK_SIZE = 512  # Bir chunk içinde indir, sonra batch'lere böl CLIP'e gönder

for chunk_start in range(0, len(to_process), CHUNK_SIZE):
    chunk = to_process.iloc[chunk_start : chunk_start + CHUNK_SIZE]
    chunk_data = list(enumerate(chunk['image_url'].values))
    chunk_ids = chunk['parent_asin'].values

    # Parallel download
    images = {}  # idx -> PIL.Image
    with ThreadPoolExecutor(max_workers=DL_WORKERS) as ex:
        futures = [ex.submit(download_image, x) for x in chunk_data]
        for fut in as_completed(futures):
            idx, img, err = fut.result()
            if img is not None:
                images[idx] = img
            else:
                new_failed_ids.append(chunk_ids[idx])

    # Batch CLIP encoding
    if images:
        success_idxs = sorted(images.keys())
        pil_list = [images[i] for i in success_idxs]

        for b in range(0, len(pil_list), BATCH_SIZE):
            batch_imgs = pil_list[b:b+BATCH_SIZE]
            batch_idx = success_idxs[b:b+BATCH_SIZE]
            inputs = clip_processor(images=batch_imgs, return_tensors='pt').to('cuda')
            with torch.no_grad():
              output = clip_model.get_image_features(**inputs)

              # transformers 5.0+ değişikliği: tensor değil, ModelOutput dönüyor
              if torch.is_tensor(output):
                emb = output  # eski davranış
              else:
                emb = output.pooler_output  # yeni davranış (5.0+)

              # L2 normalize
              emb = emb / emb.norm(p=2, dim=-1, keepdim=True)

            emb_np = emb.cpu().numpy().astype(np.float32)

            new_embeddings.append(emb_np)
            for i in batch_idx:
                new_success_ids.append(chunk_ids[i])

    processed_count += len(chunk)
    elapsed = time.time() - t0
    rate = processed_count / elapsed if elapsed > 0 else 0
    remaining = (len(to_process) - processed_count) / rate if rate > 0 else 0
    print(f'  {processed_count}/{len(to_process)} | başarı: {len(new_success_ids)} | başarısız: {len(new_failed_ids)} | hız: {rate:.1f}/s | kalan: ~{remaining/60:.1f} dk')

    # Checkpoint kaydet
    if processed_count % SAVE_EVERY < CHUNK_SIZE:
        all_emb = np.vstack([clip_embeddings_partial] + new_embeddings) if new_embeddings else clip_embeddings_partial
        all_success = successful_ids + new_success_ids
        all_failed = failed_ids + new_failed_ids
        np.savez(checkpoint_path,
                 embeddings=all_emb,
                 item_ids=np.array(all_success, dtype=object),
                 failed_ids=np.array(all_failed, dtype=object))
        print(f'Checkpoint kaydedildi ({all_emb.shape[0]} embedding)')

# Final birleştirme
if new_embeddings:
    final_embeddings = np.vstack([clip_embeddings_partial] + new_embeddings)
else:
    final_embeddings = clip_embeddings_partial
final_success = successful_ids + new_success_ids
final_failed = failed_ids + new_failed_ids

print(f'\n✓ TAMAMLANDI ({time.time()-t0:.1f}s)')
print(f'  CLIP embeddings: {final_embeddings.shape}')
print(f'  Başarılı: {len(final_success):,}')
print(f'  Başarısız: {len(final_failed):,}')

  512/13572 | başarı: 512 | başarısız: 0 | hız: 76.4/s | kalan: ~2.8 dk
  1024/13572 | başarı: 1024 | başarısız: 0 | hız: 67.7/s | kalan: ~3.1 dk
Checkpoint kaydedildi (1024 embedding)
  1536/13572 | başarı: 1536 | başarısız: 0 | hız: 66.7/s | kalan: ~3.0 dk
  2048/13572 | başarı: 2048 | başarısız: 0 | hız: 64.9/s | kalan: ~3.0 dk
Checkpoint kaydedildi (2048 embedding)
  2560/13572 | başarı: 2560 | başarısız: 0 | hız: 62.7/s | kalan: ~2.9 dk
  3072/13572 | başarı: 3072 | başarısız: 0 | hız: 61.9/s | kalan: ~2.8 dk
Checkpoint kaydedildi (3072 embedding)
  3584/13572 | başarı: 3584 | başarısız: 0 | hız: 63.2/s | kalan: ~2.6 dk
  4096/13572 | başarı: 4096 | başarısız: 0 | hız: 62.2/s | kalan: ~2.5 dk
Checkpoint kaydedildi (4096 embedding)
  4608/13572 | başarı: 4608 | başarısız: 0 | hız: 61.7/s | kalan: ~2.4 dk
  5120/13572 | başarı: 5120 | başarısız: 0 | hız: 61.4/s | kalan: ~2.3 dk
Checkpoint kaydedildi (5120 embedding)
  5632/13572 | başarı: 5631 | başarısız: 1 | hız: 56.3/s | kalan: ~

## 6. Kaydet

In [16]:
# Final CLIP embedding'leri ve ID listeleri
np.save(f'{EMBED_DIR}/clip_embeddings.npy', final_embeddings)
with open(f'{EMBED_DIR}/clip_item_ids.pkl', 'wb') as f:
    pickle.dump(final_success, f)
with open(f'{EMBED_DIR}/clip_failed_items.pkl', 'wb') as f:
    pickle.dump(final_failed, f)

# Stats
stats = {
    'sbert': {
        'shape': list(sbert_embeddings.shape),
        'n_items': int(sbert_embeddings.shape[0]),
        'dim': int(sbert_embeddings.shape[1]),
    },
    'clip': {
        'shape': list(final_embeddings.shape),
        'n_items_success': len(final_success),
        'n_items_failed': len(final_failed),
        'dim': int(final_embeddings.shape[1]) if len(final_embeddings) > 0 else 0,
        'coverage_pct': len(final_success) / len(items) * 100,
    },
}

import json
with open(f'{EMBED_DIR}/embedding_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

# Checkpoint'i temizle
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)

print('=== FINAL ===')
print(json.dumps(stats, indent=2))

print('\n Drive\'daki embeddings dosyaları:')
for fn in sorted(os.listdir(EMBED_DIR)):
    size = os.path.getsize(f'{EMBED_DIR}/{fn}') / 1e6
    print(f'  {fn}: {size:.2f} MB')

=== FINAL ===
{
  "sbert": {
    "shape": [
      13575,
      384
    ],
    "n_items": 13575,
    "dim": 384
  },
  "clip": {
    "shape": [
      13568,
      512
    ],
    "n_items_success": 13568,
    "n_items_failed": 4,
    "dim": 512,
    "coverage_pct": 99.94843462246777
  }
}

 Drive'daki embeddings dosyaları:
  clip_embeddings.npy: 27.79 MB
  clip_failed_items.pkl: 0.00 MB
  clip_item_ids.pkl: 0.18 MB
  embedding_stats.json: 0.00 MB
  sbert_embeddings.npy: 20.85 MB
  sbert_item_ids.pkl: 0.18 MB


**5 dosya `results/embeddings/` klasörüne indirilecek:**
1. `sbert_embeddings.npy`
2. `sbert_item_ids.pkl`
3. `clip_embeddings.npy`
4. `clip_item_ids.pkl`
5. `embedding_stats.json`